# Hard Case: WebDriverWait (menunggu elemen)

Di website dinamis, elemen sering **belum ada** saat halaman baru dibuka (JavaScript masih
bekerja). Kalau langsung `find_element`, hasilnya kosong/error.

Solusi pemula yang **buruk**: `time.sleep(5)` — boros waktu kalau elemen sudah siap dari
detik pertama, dan tetap gagal kalau ternyata butuh 6 detik.

Solusi yang **benar**: **`WebDriverWait`** + **`expected_conditions`** — menunggu **sampai
kondisi terpenuhi** (maksimal sekian detik), lalu lanjut begitu siap. Adaptif & efisien.

Latihan pakai `https://quotes.toscrape.com/js/` (quote-nya dirender JavaScript).

**Tooling:** `selenium` (`WebDriverWait`, `expected_conditions`).


## Contoh 1 — Menunggu elemen muncul

`WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".quote")))`
artinya: "tunggu maksimal 10 detik sampai elemen `.quote` ada di halaman, lalu kembalikan
elemen-elemennya." Kalau lewat 10 detik belum ada → `TimeoutException`.


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def buat_driver(headless=True):
    o = Options()
    if headless:  # set False kalau mau lihat browsernya
        o.add_argument("--headless=new")
    o.add_argument("--window-size=1280,900")
    return webdriver.Chrome(options=o)


driver = buat_driver()
try:
    driver.get("https://quotes.toscrape.com/js/")

    # tunggu sampai elemen .quote (hasil render JS) muncul
    quotes = WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".quote"))
    )
    print("Jumlah quote setelah ditunggu:", len(quotes))
    print("Contoh:", quotes[0].find_element(By.CSS_SELECTOR, ".text").text)
finally:
    driver.quit()


Jumlah quote setelah ditunggu: 10
Contoh: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”


## Contoh 2 — Menunggu elemen bisa diklik, lalu klik

`expected_conditions` punya banyak kondisi. Yang paling sering dipakai:

| Kondisi | Arti |
| --- | --- |
| `presence_of_element_located` | elemen sudah ada di DOM |
| `visibility_of_element_located` | elemen ada **dan** terlihat |
| `element_to_be_clickable` | elemen siap diklik (terlihat & aktif) |
| `text_to_be_present_in_element` | teks tertentu sudah muncul |
| `title_contains` | judul halaman mengandung teks |

Di bawah: tunggu tombol Next **bisa diklik** → klik → tunggu konten halaman 2 dirender.


In [2]:
driver = buat_driver()
try:
    driver.get("https://quotes.toscrape.com/js/")
    wait = WebDriverWait(driver, 10)

    # 1) pastikan quote halaman 1 sudah ada
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))
    quote_hal1 = driver.find_element(By.CSS_SELECTOR, ".quote .text").text

    # 2) tunggu tombol Next BISA DIKLIK, lalu klik
    next_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a")))
    next_btn.click()

    # 3) tunggu konten halaman 2 dirender (URL berubah ke /page/2/)
    wait.until(EC.url_contains("/page/2"))
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))
    quote_hal2 = driver.find_element(By.CSS_SELECTOR, ".quote .text").text

    print("URL sekarang :", driver.current_url)
    print("Quote hal 1  :", quote_hal1[:50], "...")
    print("Quote hal 2  :", quote_hal2[:50], "...")
finally:
    driver.quit()


URL sekarang : https://quotes.toscrape.com/js/page/2/
Quote hal 1  : “The world as we have created it is a process of o ...
Quote hal 2  : “This life is what you make it. No matter what, yo ...


## Kesimpulan & Latihan

- `WebDriverWait` + `expected_conditions` = menunggu **cerdas**, bukan `time.sleep` asal.
- Bungkus pola umum jadi fungsi helper biar dipakai berulang.
- Tangani `TimeoutException` agar program tidak mati saat elemen benar-benar tidak muncul.

**Latihan:** buat helper `ambil_teks(driver, css, timeout=10)` yang menunggu elemen muncul
lalu mengembalikan teksnya (atau `None` kalau timeout). Contoh jawaban di sel berikut.


In [3]:
# Contoh jawaban latihan
from selenium.common.exceptions import TimeoutException


def ambil_teks(driver, css, timeout=10):
    try:
        el = WebDriverWait(driver, timeout).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, css))
        )
        return el.text
    except TimeoutException:
        return None  # elemen tidak muncul dalam waktu yang ditentukan


driver = buat_driver()
try:
    driver.get("https://quotes.toscrape.com/js/")
    print("Ada       :", ambil_teks(driver, ".quote .text"))
    print("Tidak ada :", ambil_teks(driver, ".elemen-yang-tidak-ada", timeout=3))
finally:
    driver.quit()


Ada       : “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”


Tidak ada : None
